# Evaluation on Kaggle (T4 x2)

Scores the base `Qwen2.5-Coder-1.5B-Instruct` against base + LoRA adapter on the held-out
CommitBench test split, so the gain from fine-tuning is measured, not assumed. Both models
run at the same time, one per T4, in full precision (fp32), with the prompt truncation the CLI uses.

Metrics:
- **Valid format**: the prediction starts with a Conventional Commits prefix (`feat:`, `fix(x):`, ...)
- **Type accuracy**: predicted type matches the reference. Compare against the *always-majority-type*
  row: ~half the test set is `fix:`, so a model that ignores the diff already scores ~50%.
- **BLEU / ROUGE-L**: wording overlap with the human message. Useful for comparing the two models,
  not as an absolute quality score (many good messages share few words with the reference).

**Setup on Kaggle:**
1. Settings -> Accelerator -> GPU T4 x2, Internet -> On
2. Add Data -> attach the dataset containing `test.jsonl` (from `CommitBench_filter.ipynb`) and the
   dataset containing the adapter (`commit-model-lora.zip`; Kaggle unzips it). Both are found
   automatically under `/kaggle/input`.
3. Run with **Save Version -> Save & Run All (Commit)**. Rough estimate for the full ~9.7K test set
   is about an hour (both models in parallel). Set `LIMIT` for a quick check first.
   Predictions and `results.json` appear in the version's Output tab.

In [ ]:
# --- Dependencies ---
# Same torchao pin as the fine-tuning notebook.
%pip install -q -U transformers peft accelerate evaluate sacrebleu rouge_score "torchao>=0.16.0"

In [ ]:
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time
from collections import Counter

import evaluate
import pandas as pd
import torch
from huggingface_hub import snapshot_download

# --- Config & inputs ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUT_DIR = "/kaggle/working"
LIMIT = None       # e.g. 200 for a quick check; None = full test set
BATCH_SIZE = 16


def find_input(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Couldn't find {filename} anywhere under /kaggle/input. Check Add Data attached it. "
            f"Found: {glob.glob('/kaggle/input/**/*', recursive=True)[:50]}"
        )
    return matches[0]


TEST_FILE = find_input("test.jsonl")
# PEFT needs the adapter's config and weights in one folder (if not, it misreads the path as
# a Hub repo id and raises HFValidationError). Kaggle datasets may split them across
# subfolders, so gather both into one working folder.
ADAPTER_DIR = f"{OUT_DIR}/adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)
for filename in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy(find_input(filename), ADAPTER_DIR)
print("Test file:", TEST_FILE)
print("Adapter:  ", ADAPTER_DIR, os.listdir(ADAPTER_DIR))
print("GPUs:     ", torch.cuda.device_count())
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2)"

# Download once here, so the two workers don't both fetch the same weights at the same time.
snapshot_download(MODEL_NAME)


# --- Worker script: generates predictions for one model ---
# Runs as its own process so each model gets its own GPU.
WORKER_SRC = r'''
# Generates predictions for one model. Runs as its own process so each model gets its own GPU.
# Prompt shape, truncation and decoding mirror src/commit_model (cli.py / diff_utils.py).
import argparse
import json

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEQ_LEN = 768          # must match training
MAX_NEW_TOKENS = 40
PROMPT_TEMPLATE = "Write a Conventional Commits message for this diff.\n\n{diff}\n\nCommit message:"

parser = argparse.ArgumentParser()
parser.add_argument("--test-file", required=True)
parser.add_argument("--out", required=True)
parser.add_argument("--adapter", default=None)
parser.add_argument("--batch-size", type=int, default=16)
parser.add_argument("--limit", type=int, default=None)
args = parser.parse_args()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # batched generation in a decoder-only model must pad on the left

TEMPLATE_TOKENS = len(tokenizer(PROMPT_TEMPLATE.format(diff=""), add_special_tokens=False).input_ids)
DIFF_BUDGET = max(SEQ_LEN - TEMPLATE_TOKENS - MAX_NEW_TOKENS - 8, 0)


def fit_prompt(diff):
    # Truncate the diff, not the prompt's tail, so "Commit message:" always survives.
    diff_ids = tokenizer(diff, add_special_tokens=False).input_ids[:DIFF_BUDGET]
    return PROMPT_TEMPLATE.format(diff=tokenizer.decode(diff_ids))


# Full fp32, as in training: T4 has no bf16, and pure fp16 risks overflow with Qwen.
# 1.5B in fp32 is ~6GB, fine on a 16GB T4.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, device_map={"": 0})
if args.adapter:
    model = PeftModel.from_pretrained(model, args.adapter)
model.eval()

with open(args.test_file) as f:
    examples = [json.loads(line) for line in f][: args.limit]

with open(args.out, "w") as f:
    for start in range(0, len(examples), args.batch_size):
        batch = examples[start : start + args.batch_size]
        inputs = tokenizer(
            [fit_prompt(ex["diff"]) for ex in batch], return_tensors="pt", padding=True
        ).to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )
        texts = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        for ex, text in zip(batch, texts):
            pred = (text.strip().splitlines() or [""])[0].strip()
            f.write(json.dumps({"pred": pred, "ref": ex["message"]}) + "\n")
        f.flush()
        print(f"{start + len(batch)}/{len(examples)}", flush=True)
'''
with open(f"{OUT_DIR}/eval_worker.py", "w") as f:
    f.write(WORKER_SRC)


# --- Run both models in parallel, one per GPU ---
RUNS = {"base": None, "finetuned": ADAPTER_DIR}

procs = {}
for i, (name, adapter) in enumerate(RUNS.items()):
    cmd = [sys.executable, f"{OUT_DIR}/eval_worker.py", "--test-file", TEST_FILE,
           "--out", f"{OUT_DIR}/preds_{name}.jsonl", "--batch-size", str(BATCH_SIZE)]
    if adapter:
        cmd += ["--adapter", adapter]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    # With one GPU both share it (~6GB each in fp32, fits in 16GB).
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i % torch.cuda.device_count())}
    log_path = f"{OUT_DIR}/log_{name}.txt"
    procs[name] = (subprocess.Popen(cmd, env=env, stdout=open(log_path, "w"), stderr=subprocess.STDOUT), log_path)


def last_line(path):
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]
    return lines[-1] if lines else "(starting)"


started = time.time()
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(60)
    status = " | ".join(f"{name}: {last_line(log)}" for name, (_, log) in procs.items())
    print(f"[{(time.time() - started) / 60:5.1f} min] {status}", flush=True)

for name, (p, log) in procs.items():
    if p.returncode != 0:
        with open(log) as f:
            raise RuntimeError(f"{name} failed (exit {p.returncode}):\n" + "".join(f.readlines()[-40:]))
print(f"Done in {(time.time() - started) / 60:.1f} min")


# --- Metrics ---
TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")
VALID_RE = re.compile(r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S")


def extract_type(message):
    match = TYPE_RE.match(message.strip())
    return match.group(1) if match else None


bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

preds = {}
for name in RUNS:
    with open(f"{OUT_DIR}/preds_{name}.jsonl") as f:
        preds[name] = [json.loads(line) for line in f]

refs = [r["ref"] for r in preds["base"]]
ref_types = [extract_type(r) for r in refs]
majority_type, majority_count = Counter(ref_types).most_common(1)[0]

rows = [{"model": f"always '{majority_type}:'", "type_acc": majority_count / len(refs)}]
for name, records in preds.items():
    p = [r["pred"] for r in records]
    rows.append({
        "model": name,
        "valid_format": sum(bool(VALID_RE.match(x)) for x in p) / len(p),
        "type_acc": sum(extract_type(x) == t for x, t in zip(p, ref_types)) / len(p),
        "BLEU": bleu.compute(predictions=p, references=[[r] for r in refs])["score"],
        "ROUGE-L": rouge.compute(predictions=p, references=refs)["rougeL"],
    })

results = pd.DataFrame(rows).set_index("model")
print(f"Examples: {len(refs)}\n")
print(results.to_string(float_format=lambda v: f"{v:.3f}", na_rep="-"))
results.to_json(f"{OUT_DIR}/results.json", orient="index", indent=2)

print("\nPer-type accuracy (finetuned):")
per_type = Counter(ref_types)
for t, n in per_type.most_common():
    hits = sum(extract_type(r["pred"]) == t for r, rt in zip(preds["finetuned"], ref_types) if rt == t)
    print(f"  {t:<9} {hits / n:6.1%}  (n={n})")

print("\nSample predictions:")
for i in range(10):
    print(f"  ref:       {refs[i]}")
    print(f"  base:      {preds['base'][i]['pred']}")
    print(f"  finetuned: {preds['finetuned'][i]['pred']}\n")
